# 1. DATA CLEANING
## HairBright · Marketing Mix Modeling — Revenue Prediction

---

**Input:** `../data/raw/HairBright_mmm_haircare_data.xlsx`  
*Daily advertising and revenue data from Google Ads and Meta Ads — Beauty & Fitness / Hair Care vertical, United States market.*

**Output:** `../data/interim/hairbright_clean_YYYYMMDD.xlsx`  
*Weekly-aggregated, cleaned and normalized dataset ready for EDA and MMM transformation.*

---
## 1.1. STARTING SITUATION

HairBright is a hair care brand that invests a significant portion of its monthly advertising budget in **Google Ads** and **Meta Ads** campaigns.

The investment is distributed across multiple channels (Paid Search, Performance Max, Display, Video, Facebook and Instagram), ad formats and audiences, generating highly variable returns.

The large number of variables affecting performance is causing HairBright to face the following challenges:

- A global ROAS below that of its main competitors in the Hair Care category.
- Difficulty in granularly identifying the segments (channels, audiences, seasonal moments) with the highest and lowest potential, which prevents efficient optimization of the advertising budget.

An improvement in return on investment would allow HairBright to profitably increase its monthly budget and gain market share in the competitive hair care sector in the United States.

Campaign data is downloaded monthly and consolidated into a single dataset. Data is currently available from **July 2022 to December 2023 (74 weeks)**, with detail on pre-click investment (impressions, clicks and cost) and post-click data on conversions, revenue and ROAS provided by the marketing team at the end of each month.

### 1.1.1. Dataset origin

The dataset used in this project is based on real-world marketing and sales data originally published on Figshare under the title:

> *"Multi-Region Marketing Mix Modelling (MMM) Dataset for Several eCommerce Brands"*  
> Anderson, 2024 — DOI: [10.6084/m9.figshare.25314841](https://figshare.com/articles/dataset/Multi-Region_Marketing_Mix_Modeling_MMM_Dataset_for_Several_eCommerce_Brands/25314841)

**Key characteristics of the source dataset:**

- Contains anonymized daily data from close to 100 eCommerce brands across multiple regions.
- Includes pre-click metrics (impressions, clicks, spend) from Google and Meta, and post-click performance (orders, revenue, units sold, discounts).
- All brand names and company identifiers have been anonymized to protect confidentiality.

**Scope applied for this project:**

| Dimension | Selection |
|:----------|:----------|
| Vertical | Beauty & Fitness — Hair Care |
| Brand | HairBright (anonymized) |
| Market | United States |
| Granularity | Weekly (aggregated from daily) |
| Channels | Google Ads + Meta Ads |

**Data flow summary:**

- **Source:** Public Figshare repository (real anonymized eCommerce MMM dataset).
- **Pre-click:** Advertising spend, impressions and clicks from Google (Paid Search, Shopping, PMax, Display, Video) and Meta (Facebook, Instagram, Other).
- **Post-click:** Revenue, orders (total and new customers) and units sold, provided monthly by the brand's internal tracking.
- **Processing:** Daily data → weekly aggregation + normalization of sub-unit scaled fields + preparation of derived metrics for MMM.

---
## 1.2. MODEL OBJECTIVE

- **Business objective:** Optimize HairBright's advertising budget allocation to maximize revenue generated per dollar invested.
- **Model target:** Predict weekly **revenue in USD** — both at a global level and broken down by media channel, platform and time period.
- **Why revenue is a better objective than ROAS — even beyond technical constraints:**

  The intuitive choice might be to optimise ROAS directly, but revenue is the superior modelling objective for both technical and strategic reasons:

  *Technical reasons (hard constraints):*
  - **Endogeneity**: ROAS = Revenue / Spend. Spend appears in both the target and the predictors, biasing all estimated coefficients by construction.
  - **Heteroscedasticity**: in low-investment weeks, ROAS becomes unstable (e.g. $50k revenue on $488 spend = 103x ROAS). The variance of the target is not homogeneous, violating OLS assumptions and making regularization unreliable.

  *Strategic reasons (why revenue is genuinely better, not just a workaround):*
  - **Scale-sensitivity**: maximising ROAS can lead to recommending minimal spend (a €1 investment with €2 return gives 200% ROAS but creates no business value). Revenue as the target keeps the model aligned with actual growth.
  - **Additivity**: revenue contributions from different channels can be summed to a meaningful total. Channel-level ROAS values cannot be averaged or aggregated without weighting by spend, making budget decisions harder to operationalise.
  - **Richer decomposition**: modelling revenue allows the MMM to separate baseline revenue (organic, seasonality, CRM) from incremental media contribution — a decomposition that has no clean equivalent when ROAS is the target.
  - **ROAS is recovered as output, not lost**: by dividing each channel's estimated revenue contribution by its spend, we obtain a channel-level ROAS that is *cleaner* than a directly modelled one, because it is incremental and free from the confounds listed above.

  **Revenue is therefore not a compromise — it is the correct primary objective.** ROAS remains the business KPI and is reported as a derived metric.
- **Impact (Value):** Channel-level revenue contribution estimates enable budget reallocation towards the highest-return segments, increasing overall ROAS and allowing HairBright to scale investment with measurable confidence.

**The model must be explainable** to build trust with the marketing team and guide business strategy. To this end, it must clearly answer the following key questions:

- **Which media mix to prioritize?** Ranking channels by estimated revenue contribution and derived ROAS.
- **What factors influence revenue?** Using MMM coefficients and SHAP values to quantify, for example, why Performance Max generates higher returns than Paid Search, or why Black Friday multiplies performance independently of media spend.

---
## 1.3. INITIAL SETUP

We import the necessary libraries and establish the connection with Google Drive.

**Libraries used:**
- `pandas`: Data manipulation and aggregation
- `numpy`: Numerical operations and normalization
- `pathlib`: File path management
- `datetime`: Filename timestamping
- `warnings`: Suppress non-critical warnings

In [1]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Pandas display configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.4f}'.format)

print(f"Libraries loaded successfully")
print(f"Pandas version: {pd.__version__}")

Libraries loaded successfully
Pandas version: 2.2.2


In [2]:
# Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Project paths
PATH_PROJECT = Path(
    '/content/drive/MyDrive/PRO/+ DATA SCIENCE/TFM/'
    'marketing-mix-modeling-beauty'
)
PATH_DATA_RAW     = PATH_PROJECT / 'data' / 'raw'
PATH_DATA_INTERIM = PATH_PROJECT / 'data' / 'interim'

PATH_DATA_INTERIM.mkdir(parents=True, exist_ok=True)

print("Project paths:")
print(f"  Project : {PATH_PROJECT}")
print(f"  Raw     : {PATH_DATA_RAW}")
print(f"  Interim : {PATH_DATA_INTERIM}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project paths:
  Project : /content/drive/MyDrive/PRO/+ DATA SCIENCE/TFM/marketing-mix-modeling-beauty
  Raw     : /content/drive/MyDrive/PRO/+ DATA SCIENCE/TFM/marketing-mix-modeling-beauty/data/raw
  Interim : /content/drive/MyDrive/PRO/+ DATA SCIENCE/TFM/marketing-mix-modeling-beauty/data/interim


---
## 1.4. PRELIMINARY DATASET ANALYSIS

Before loading the data, we inspect the raw Excel file to identify:

- File structure (sheets, header rows, available columns)
- Data types requiring coercion
- Scaling anomalies in monetary fields
- Potential error values or corrupted cells

The raw dataset contains **509 daily rows × 50 columns** spanning July 2022 to December 2023. After weekly aggregation the working dataset will contain **74 weeks**.

### 1.4.1. Known issues and transformations to apply

| # | Problem | Affected columns | Description | Solution |
|:--|:--------|:-----------------|:------------|:---------|
| 1 | Sub-unit scaling | `ALL_PURCHASES_ORIGINAL_PRICE`, `FIRST_PURCHASES_ORIGINAL_PRICE`, `*_GROSS_DISCOUNT` | Values stored at sub-unit precision (e.g. `4,552,984,240,467` instead of `4,552.98 USD`). Confirmed normalization factor: **÷ 1,000,000,000** | Divide by `1e9` after coercion to float (step 1.6.4) |
| 2 | Bimodal spend scale | All `*_SPEND` columns except `GOOGLE_VIDEO_SPEND` | Some daily rows stored at sub-unit precision (value ≥ 1,000,000), others at correct USD scale. Conditional ÷ 1e9 applied per row | Normalize only rows ≥ 1,000,000 after all columns are coerced to float (step 1.6.4) |
| 3 | Date values in spend column | `GOOGLE_PAID_SEARCH_SPEND` | 7 rows where Excel parsed the spend value as a date object (e.g. `2026-07-22`) due to cell formatting error in source file | Coerce to numeric with `errors='coerce'` → impute with column median (step 1.6.2) |
| 4 | Null spend values | All `*_SPEND` columns | Null = no investment that day for that channel. Not a data error | Coerced to `float64` and filled with `0` during numeric coercion (step 1.6.3) |
| 5 | Channels with 100% nulls | `GOOGLE_SHOPPING_SPEND`, `TIKTOK_SPEND` | No investment recorded across the entire period | Drop from model features (step 1.6.1) |
| 6 | Daily → weekly aggregation | All columns | Raw data is daily; MMM requires weekly granularity to capture adstock effects | Sum spend/revenue; mean for ratios; week start = Monday (step 1.6.5) |
| 7 | Metadata columns | `MMM_TIMESERIES_ID`, `ORGANISATION_ID`, etc. | Identifiers with no predictive value | Drop before modelling (step 1.6.1) |


---
## 1.5. DATA LOADING

We load the raw Excel file. No header skipping is required — the first row contains column names directly.

In [3]:
# Load raw data
FILE_RAW = PATH_DATA_RAW / 'HairBright_mmm_haircare_data.xlsx'

df_raw = pd.read_excel(FILE_RAW)
initial_records = len(df_raw)

print(f"File loaded  : {FILE_RAW.name}")
print(f"Dimensions   : {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
print(f"Date range   : {df_raw['DATE_DAY'].min().date()} → {df_raw['DATE_DAY'].max().date()}")
print(f"Unique weeks : {df_raw['DATE_DAY'].dt.to_period('W').nunique()}")

File loaded  : HairBright_mmm_haircare_data.xlsx
Dimensions   : 509 rows × 50 columns
Date range   : 2022-07-29 → 2023-12-19
Unique weeks : 74


In [4]:
# Inspect original columns and dtypes
print("Original columns and data types:")
print("-" * 55)
for i, (col, dtype) in enumerate(zip(df_raw.columns, df_raw.dtypes), 1):
    print(f"  {i:2d}. {col:<45} {dtype}")

Original columns and data types:
-------------------------------------------------------
   1. MMM_TIMESERIES_ID                             object
   2. ORGANISATION_ID                               object
   3. ORGANISATION_VERTICAL                         object
   4. ORGANISATION_SUBVERTICAL                      object
   5. ORGANISATION_MARKETING_SOURCES                object
   6. ORGANISATION_PRIMARY_TERRITORY_NAME           object
   7. TERRITORY_NAME                                object
   8. DATE_DAY                                      datetime64[ns]
   9. CURRENCY_CODE                                 object
  10. FIRST_PURCHASES                               int64
  11. FIRST_PURCHASES_UNITS                         int64
  12. FIRST_PURCHASES_ORIGINAL_PRICE                object
  13. FIRST_PURCHASES_GROSS_DISCOUNT                object
  14. ALL_PURCHASES                                 int64
  15. ALL_PURCHASES_UNITS                           int64
  16. ALL_PURCHASES_OR

---
## 1.6. DATA CLEANING

### 1.6.1. Drop metadata and zero-coverage columns

Columns that carry no predictive signal are removed before processing:
- **Metadata identifiers:** `MMM_TIMESERIES_ID`, `ORGANISATION_ID`, `ORGANISATION_VERTICAL`, `ORGANISATION_SUBVERTICAL`, `ORGANISATION_MARKETING_SOURCES`, `ORGANISATION_PRIMARY_TERRITORY_NAME`, `TERRITORY_NAME`, `CURRENCY_CODE`
- **Zero-investment channels:** `GOOGLE_SHOPPING_*`, `TIKTOK_*` — 100% null across the full period, confirming no spend was made.

In [5]:
# Working copy
df = df_raw.copy()

# Metadata columns — no predictive value
META_COLS = [
    'MMM_TIMESERIES_ID', 'ORGANISATION_ID', 'ORGANISATION_VERTICAL',
    'ORGANISATION_SUBVERTICAL', 'ORGANISATION_MARKETING_SOURCES',
    'ORGANISATION_PRIMARY_TERRITORY_NAME', 'TERRITORY_NAME', 'CURRENCY_CODE'
]

# Zero-coverage channels (100% null across full period)
ZERO_COVERAGE = [col for col in df.columns if 'SHOPPING' in col or 'TIKTOK' in col]

cols_to_drop = META_COLS + ZERO_COVERAGE
df = df.drop(columns=cols_to_drop)

print(f"Columns dropped : {len(cols_to_drop)}")
print(f"  Metadata      : {len(META_COLS)}")
print(f"  Zero-coverage : {len(ZERO_COVERAGE)} ({', '.join(ZERO_COVERAGE[:4])}...)")
print(f"Remaining cols  : {df.shape[1]}")

Columns dropped : 14
  Metadata      : 8
  Zero-coverage : 6 (GOOGLE_SHOPPING_SPEND, TIKTOK_SPEND, GOOGLE_SHOPPING_CLICKS, TIKTOK_CLICKS...)
Remaining cols  : 36


### 1.6.2. Fix corrupted values in `GOOGLE_PAID_SEARCH_SPEND`

Seven rows contain date objects instead of numeric spend values — a cell formatting error in the source Excel export. These are coerced to `NaN` and subsequently imputed with the column **median** (robust to outliers) to preserve the weekly aggregation.

In [6]:
# Identify corrupted rows (date objects in a numeric column)
ps_numeric = pd.to_numeric(df['GOOGLE_PAID_SEARCH_SPEND'], errors='coerce')
error_mask = ps_numeric.isna() & df['GOOGLE_PAID_SEARCH_SPEND'].notna()
n_errors = error_mask.sum()

print(f"Corrupted rows found in GOOGLE_PAID_SEARCH_SPEND: {n_errors}")
print()
print("Corrupted values:")
print(df[error_mask][['DATE_DAY', 'GOOGLE_PAID_SEARCH_SPEND']].to_string(index=False))

# Fix: coerce to numeric (NaN for errors), then impute with median
# IMPORTANT: compute median only from correct-scale rows (< 1e6 USD)
# to avoid imputing with a sub-unit-scaled value.
df['GOOGLE_PAID_SEARCH_SPEND'] = ps_numeric
correct_scale = df['GOOGLE_PAID_SEARCH_SPEND'][df['GOOGLE_PAID_SEARCH_SPEND'] < 1e6]
col_median = correct_scale.median()
df['GOOGLE_PAID_SEARCH_SPEND'] = df['GOOGLE_PAID_SEARCH_SPEND'].fillna(col_median)

print(f"\nImputed with correct-scale median: ${col_median:.2f}")
print(f"Remaining nulls    : {df['GOOGLE_PAID_SEARCH_SPEND'].isna().sum()} ✓")

Corrupted rows found in GOOGLE_PAID_SEARCH_SPEND: 7

Corrupted values:
  DATE_DAY GOOGLE_PAID_SEARCH_SPEND
2022-08-24      2026-07-22 00:00:00
2023-06-12      2026-04-23 00:00:00
2023-07-23      2026-03-22 00:00:00
2023-08-02      2026-09-12 00:00:00
2023-08-24      2026-06-25 00:00:00
2023-08-25      2026-07-05 00:00:00
2023-09-13      2026-02-03 00:00:00

Imputed with correct-scale median: $62.26
Remaining nulls    : 0 ✓


### 1.6.3. Coerce all numeric columns to float64

All spend, clicks, impressions and monetary columns are cast to `float64` before any scale correction is applied.

Ensuring a uniform numeric dtype at this stage guarantees that the threshold comparison (`>= 1,000,000`) used in the next step works correctly across every spend column, regardless of how the source Excel file stored the original values. Non-numeric residual values are coerced to `NaN` and replaced with `0` (no activity that day for that channel).

**Columns covered by this step:**

| Pattern | Column type |
|:--------|:------------|
| `*_SPEND` | Advertising spend per channel |
| `*_CLICKS` | Clicks per channel |
| `*_IMPRESSIONS` | Impressions per channel |
| `*_PURCHASES*`, `*_UNITS*` | Orders and units sold |
| `*_DISCOUNT*` | Gross discount amounts |
| `DIRECT_*`, `BRANDED_*`, `ORGANIC_*`, `EMAIL_*`, `REFERRAL_*`, `ALL_OTHER_*` | Attribution click sources |


In [7]:
# Columns to cast to float64 — covers spend, clicks, impressions and all monetary fields
NUMERIC_COLS = [col for col in df.columns
                if any(x in col for x in [
                    'SPEND', 'CLICKS', 'IMPRESSIONS',
                    'PURCHASES', 'DISCOUNT', 'UNITS',
                    'DIRECT', 'BRANDED', 'ORGANIC',
                    'EMAIL', 'REFERRAL', 'OTHER'
                ])]

for col in NUMERIC_COLS:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

remaining_nulls = df[NUMERIC_COLS].isnull().sum().sum()
print(f"Numeric columns coerced : {len(NUMERIC_COLS)}")
print(f"Remaining nulls         : {remaining_nulls} ✓")
print(f"All columns now float64 — scale correction can proceed safely")


Numeric columns coerced : 35
Remaining nulls         : 0 ✓
All columns now float64 — scale correction can proceed safely


### 1.6.4. Normalize sub-unit scaled monetary fields

Four revenue/discount columns and six spend columns contain values at sub-unit precision — stored as integers scaled by **÷ 1,000,000,000** relative to the true USD amounts. This is a known characteristic of the Figshare source dataset (documented in 1.1.1).

All columns are guaranteed to be `float64` by step 1.6.3, so the threshold comparison and division are both numerically exact.

**Revenue and discount columns** — every row is in sub-unit form; apply ÷ 1e9 unconditionally:

| Raw value | Normalized (÷ 1e9) | Interpretation |
|----------:|-------------------:|:---------------|
| 4,552,984,240,467 | 4,552.98 | ~$4,553 USD/day ✓ |
| 317,799,803,177 | 317.80 | ~$318 USD/day ✓ |

**Spend columns** — bimodal distribution: some daily rows at correct USD scale, others at sub-unit scale. A conditional ÷ 1e9 is applied **per row** (only where value ≥ 1,000,000), leaving already-correct rows unchanged.

`GOOGLE_VIDEO_SPEND` is excluded — all its values are in correct USD scale.


In [8]:
# ── Revenue and discount columns — every row is in sub-unit form ────────────
SUB_UNIT_COLS = [
    'ALL_PURCHASES_ORIGINAL_PRICE',
    'FIRST_PURCHASES_ORIGINAL_PRICE',
    'ALL_PURCHASES_GROSS_DISCOUNT',
    'FIRST_PURCHASES_GROSS_DISCOUNT'
]
SCALE_FACTOR = 1e9

for col in SUB_UNIT_COLS:
    df[col] = df[col] / SCALE_FACTOR

# Verify normalization
print("Post-normalization revenue statistics:")
print(df['ALL_PURCHASES_ORIGINAL_PRICE'].describe().apply(lambda x: f"${x:,.2f}"))
print()
print("Daily revenue range: ${:,.0f} – ${:,.0f}  →  plausible for US hair care e-commerce ✓".format(
    df['ALL_PURCHASES_ORIGINAL_PRICE'].min(),
    df['ALL_PURCHASES_ORIGINAL_PRICE'].max()
))

# ── Spend columns — conditional ÷ 1e9 (rows >= 1,000,000 only) ──────────────
# All columns are already float64 (step 1.6.3).
# Rows with value >= 1e6 are in sub-unit scale; rows < 1e6 are already correct USD.
# GOOGLE_VIDEO_SPEND is excluded — all its values are in correct USD scale.
SPEND_COLS_TO_FIX = [
    'GOOGLE_PAID_SEARCH_SPEND', 'GOOGLE_PMAX_SPEND', 'GOOGLE_DISPLAY_SPEND',
    'META_FACEBOOK_SPEND', 'META_INSTAGRAM_SPEND', 'META_OTHER_SPEND'
]
SPEND_SCALE_THRESHOLD = 1e6

spend_rows_fixed = {}
for col in SPEND_COLS_TO_FIX:
    if col in df.columns:
        mask = df[col] >= SPEND_SCALE_THRESHOLD
        spend_rows_fixed[col] = mask.sum()
        df.loc[mask, col] = df.loc[mask, col] / 1e9

print("\nSpend scale correction (daily rows ÷1e9 per column):")
for col, n in spend_rows_fixed.items():
    print(f"  {col:<35} {n:>3} rows fixed")
print(f"\nPost-fix GOOGLE_PAID_SEARCH_SPEND range: "
      f"${df['GOOGLE_PAID_SEARCH_SPEND'].min():.2f} – "
      f"${df['GOOGLE_PAID_SEARCH_SPEND'].max():.2f}  ✓")


Post-normalization revenue statistics:
count        $509.00
mean       $8,218.35
std       $10,855.93
min            $0.00
25%        $4,360.99
50%        $6,333.99
75%        $9,177.15
max      $140,167.67
Name: ALL_PURCHASES_ORIGINAL_PRICE, dtype: object

Daily revenue range: $0 – $140,168  →  plausible for US hair care e-commerce ✓

Spend scale correction (daily rows ÷1e9 per column):
  GOOGLE_PAID_SEARCH_SPEND            263 rows fixed
  GOOGLE_PMAX_SPEND                    74 rows fixed
  GOOGLE_DISPLAY_SPEND                  4 rows fixed
  META_FACEBOOK_SPEND                  77 rows fixed
  META_INSTAGRAM_SPEND                 24 rows fixed
  META_OTHER_SPEND                     18 rows fixed

Post-fix GOOGLE_PAID_SEARCH_SPEND range: $0.00 – $662.46  ✓


### 1.6.5. Weekly aggregation (daily → weekly)

The MMM requires weekly granularity to:
- Capture advertising carryover (adstock) effects that span multiple days
- Reduce daily noise while preserving seasonal patterns
- Align with the 74-week analytical window

**Aggregation rules:**

| Column type | Aggregation |
|:------------|:------------|
| Revenue, spend, clicks, impressions, orders, units | `sum` |
| Week index | First day of the week (Monday) |

In [9]:
# Parse date and set week start to Monday
df['DATE_DAY'] = pd.to_datetime(df['DATE_DAY'])
df['WEEK'] = df['DATE_DAY'].dt.to_period('W').dt.start_time

# Define aggregation rules: all numeric columns → sum
agg_dict = {col: 'sum' for col in df.select_dtypes(include='number').columns}

# Aggregate daily → weekly
df_weekly = df.groupby('WEEK').agg(agg_dict).reset_index()

print(f"Aggregation complete:")
print(f"  Daily rows   : {len(df)}")
print(f"  Weekly rows  : {len(df_weekly)}")
print(f"  Week range   : {df_weekly['WEEK'].min().date()} → {df_weekly['WEEK'].max().date()}")
print(f"  Columns      : {df_weekly.shape[1]}")

Aggregation complete:
  Daily rows   : 509
  Weekly rows  : 74
  Week range   : 2022-07-25 → 2023-12-18
  Columns      : 36


### 1.6.6. Rename columns to clean snake_case

Column names are standardized to lowercase snake_case for readability and downstream compatibility with Python modelling libraries.

In [10]:
# Column renaming map
RENAME_MAP = {
    'WEEK'                              : 'week',
    'ALL_PURCHASES_ORIGINAL_PRICE'      : 'revenue',
    'FIRST_PURCHASES_ORIGINAL_PRICE'    : 'revenue_new_customers',
    'ALL_PURCHASES_GROSS_DISCOUNT'      : 'discount_total',
    'FIRST_PURCHASES_GROSS_DISCOUNT'    : 'discount_new_customers',
    'ALL_PURCHASES'                     : 'orders_total',
    'ALL_PURCHASES_UNITS'               : 'units_total',
    'FIRST_PURCHASES'                   : 'orders_new_customers',
    'FIRST_PURCHASES_UNITS'             : 'units_new_customers',
    'GOOGLE_PAID_SEARCH_SPEND'          : 'spend_ps',
    'GOOGLE_PMAX_SPEND'                 : 'spend_pmax',
    'GOOGLE_DISPLAY_SPEND'              : 'spend_display',
    'GOOGLE_VIDEO_SPEND'                : 'spend_video',
    'META_FACEBOOK_SPEND'               : 'spend_fb',
    'META_INSTAGRAM_SPEND'              : 'spend_ig',
    'META_OTHER_SPEND'                  : 'spend_meta_other',
    'GOOGLE_PAID_SEARCH_CLICKS'         : 'clicks_ps',
    'GOOGLE_PMAX_CLICKS'                : 'clicks_pmax',
    'GOOGLE_DISPLAY_CLICKS'             : 'clicks_display',
    'GOOGLE_VIDEO_CLICKS'               : 'clicks_video',
    'META_FACEBOOK_CLICKS'              : 'clicks_fb',
    'META_INSTAGRAM_CLICKS'             : 'clicks_ig',
    'META_OTHER_CLICKS'                 : 'clicks_meta_other',
    'GOOGLE_PAID_SEARCH_IMPRESSIONS'    : 'impr_ps',
    'GOOGLE_PMAX_IMPRESSIONS'           : 'impr_pmax',
    'GOOGLE_DISPLAY_IMPRESSIONS'        : 'impr_display',
    'GOOGLE_VIDEO_IMPRESSIONS'          : 'impr_video',
    'META_FACEBOOK_IMPRESSIONS'         : 'impr_fb',
    'META_INSTAGRAM_IMPRESSIONS'        : 'impr_ig',
    'META_OTHER_IMPRESSIONS'            : 'impr_meta_other',
    'DIRECT_CLICKS'                     : 'clicks_direct',
    'BRANDED_SEARCH_CLICKS'             : 'clicks_branded',
    'ORGANIC_SEARCH_CLICKS'             : 'clicks_organic',
    'EMAIL_CLICKS'                      : 'clicks_email',
    'REFERRAL_CLICKS'                   : 'clicks_referral',
    'ALL_OTHER_CLICKS'                  : 'clicks_other',
}

df_weekly = df_weekly.rename(columns=RENAME_MAP)

# Drop any remaining columns not in rename map (safety)
final_cols = list(RENAME_MAP.values())
df_weekly = df_weekly[[c for c in final_cols if c in df_weekly.columns]]

print(f"Columns after renaming: {df_weekly.shape[1]}")
print("\nFinal column list:")
for i, col in enumerate(df_weekly.columns, 1):
    print(f"  {i:2d}. {col}")

Columns after renaming: 36

Final column list:
   1. week
   2. revenue
   3. revenue_new_customers
   4. discount_total
   5. discount_new_customers
   6. orders_total
   7. units_total
   8. orders_new_customers
   9. units_new_customers
  10. spend_ps
  11. spend_pmax
  12. spend_display
  13. spend_video
  14. spend_fb
  15. spend_ig
  16. spend_meta_other
  17. clicks_ps
  18. clicks_pmax
  19. clicks_display
  20. clicks_video
  21. clicks_fb
  22. clicks_ig
  23. clicks_meta_other
  24. impr_ps
  25. impr_pmax
  26. impr_display
  27. impr_video
  28. impr_fb
  29. impr_ig
  30. impr_meta_other
  31. clicks_direct
  32. clicks_branded
  33. clicks_organic
  34. clicks_email
  35. clicks_referral
  36. clicks_other


### 1.6.7. Add time flags

Binary and categorical time flags are added to capture seasonality and known performance multipliers. These will be used as control variables in the MMM.

In [11]:
# Time-based feature flags
df_weekly['year']       = df_weekly['week'].dt.year
df_weekly['month']      = df_weekly['week'].dt.month
df_weekly['week_num']   = df_weekly['week'].dt.isocalendar().week.astype(int)
df_weekly['quarter']    = df_weekly['week'].dt.quarter

# Q4 flag (Oct–Dec): highest revenue period for US e-commerce
df_weekly['is_q4'] = (df_weekly['month'].isin([10, 11, 12])).astype(int)

# Black Friday week flag (week containing the 4th Thursday of November)
def is_black_friday_week(week_start):
    # Black Friday = day after 4th Thursday of November
    year = week_start.year
    nov_thursdays = pd.date_range(f'{year}-11-01', f'{year}-11-30', freq='W-THU')
    bf = nov_thursdays[3] + pd.Timedelta(days=1)  # Friday after 4th Thursday
    week_end = week_start + pd.Timedelta(days=6)
    return int(week_start <= bf <= week_end)

df_weekly['is_bf_week'] = df_weekly['week'].apply(is_black_friday_week)

# Holiday flag (Christmas + New Year weeks)
holiday_weeks = df_weekly['week'].apply(
    lambda w: int(w.month == 12 and w.day >= 18 or w.month == 1 and w.day <= 7)
)
df_weekly['is_holiday'] = holiday_weeks

print("Time flags added:")
print(f"  is_q4     : {df_weekly['is_q4'].sum()} weeks")
print(f"  is_bf_week: {df_weekly['is_bf_week'].sum()} weeks")
print(f"  is_holiday: {df_weekly['is_holiday'].sum()} weeks")

Time flags added:
  is_q4     : 25 weeks
  is_bf_week: 2 weeks
  is_holiday: 4 weeks


---
## 1.7. CLEAN DATASET VALIDATION

### 1.7.1. Final dimensions

In [12]:
print(f"Shape       : {df_weekly.shape[0]} rows × {df_weekly.shape[1]} columns")
pct = (df_weekly.shape[0] / 74) * 100
print(f"Week coverage : {pct:.1f}% of expected 74 weeks")
print(f"Date range  : {df_weekly['week'].min().date()} → {df_weekly['week'].max().date()}")

Shape       : 74 rows × 43 columns
Week coverage : 100.0% of expected 74 weeks
Date range  : 2022-07-25 → 2023-12-18


### 1.7.2. Null and duplicate verification

In [13]:
# Nulls
nulls = df_weekly.isnull().sum().sum()
print(f"Null values     : {nulls}" + (" ✓" if nulls == 0 else " ⚠"))

# Duplicates
duplicates = df_weekly.duplicated(subset='week').sum()
print(f"Duplicate weeks : {duplicates}" + (" ✓" if duplicates == 0 else " ⚠"))

# Nulls by column (only show if any)
col_nulls = df_weekly.isnull().sum()
if col_nulls.sum() > 0:
    print("\nColumns with nulls:")
    print(col_nulls[col_nulls > 0])

Null values     : 0 ✓
Duplicate weeks : 0 ✓


### 1.7.3. Revenue sanity check

In [14]:
# Weekly revenue statistics
print("Weekly revenue (USD) — descriptive statistics:")
print(df_weekly['revenue'].describe().apply(lambda x: f"${x:>12,.2f}"))
print()

# Flag suspiciously low revenue weeks
low_rev = df_weekly[df_weekly['revenue'] < 1000]
if len(low_rev) > 0:
    print(f"⚠  Weeks with revenue < $1,000: {len(low_rev)}")
    print(low_rev[['week', 'revenue']].to_string(index=False))
else:
    print("All weeks have revenue > $1,000 ✓")

Weekly revenue (USD) — descriptive statistics:
count    $       74.00
mean     $   56,528.93
std      $   58,461.22
min      $   10,339.78
25%      $   33,710.22
50%      $   42,744.61
75%      $   59,216.81
max      $  474,930.36
Name: revenue, dtype: object

All weeks have revenue > $1,000 ✓


### 1.7.4. Spend coverage by channel

In [15]:
# Spend columns
spend_cols = [c for c in df_weekly.columns if c.startswith('spend_')]

print(f"{'Channel':<22} {'Active weeks':>14} {'Coverage':>10} {'Weekly avg ($)':>16} {'Total ($)':>14}")
print("-" * 80)
for col in spend_cols:
    active = (df_weekly[col] > 0).sum()
    coverage = active / len(df_weekly) * 100
    avg = df_weekly[df_weekly[col] > 0][col].mean()
    total = df_weekly[col].sum()
    print(f"{col:<22} {active:>14} {coverage:>9.1f}% {avg:>16,.2f} {total:>14,.2f}")

Channel                  Active weeks   Coverage   Weekly avg ($)      Total ($)
--------------------------------------------------------------------------------
spend_ps                           74     100.0%           512.03      37,890.03
spend_pmax                         73      98.6%         2,538.52     185,312.17
spend_display                      14      18.9%           146.40       2,049.58
spend_video                         7       9.5%           111.36         779.50
spend_fb                           48      64.9%         1,974.07      94,755.46
spend_ig                           11      14.9%           574.69       6,321.56
spend_meta_other                   11      14.9%            10.65         117.16


### 1.7.5. Preview of clean weekly dataset

In [16]:
df_weekly.head(10)

,week,revenue,revenue_new_customers,discount_total,discount_new_customers,orders_total,units_total,orders_new_customers,units_new_customers,spend_ps,spend_pmax,spend_display,spend_video,spend_fb,spend_ig,spend_meta_other,clicks_ps,clicks_pmax,clicks_display,clicks_video,clicks_fb,clicks_ig,clicks_meta_other,impr_ps,impr_pmax,impr_display,impr_video,impr_fb,impr_ig,impr_meta_other,clicks_direct,clicks_branded,clicks_organic,clicks_email,clicks_referral,clicks_other,year,month,week_num,quarter,is_q4,is_bf_week,is_holiday
0,2022-07-25,"10,339.7829","11,153.9783","2,965.3976","2,261.7900",83,106,67,87,186.7800,910.0474,0.0000,0.0000,757.1000,0.0000,0.0000,0.0000,"1,008.0000",0.0000,0.0000,439.0000,0.0000,0.0000,0.0000,"134,563.0000",0.0000,0.0000,"63,903.0000",0.0000,0.0000,610.0000,0.0000,"1,169.0000",640.0000,279.0000,167,2022,7,30,3,0,0,0
1,2022-08-01,"31,882.8654","22,860.6154","6,072.9994","5,494.1647",196,260,174,230,176.9100,"2,225.2569",0.0000,0.0000,"1,299.8100",0.0000,0.0000,216.0000,"2,414.0000",0.0000,0.0000,637.0000,0.0000,0.0000,"3,459.0000","410,154.0000",0.0000,0.0000,"106,337.0000",0.0000,0.0000,"1,512.0000",0.0000,"3,105.0000",705.0000,884.0000,396,2022,8,31,3,0,0,0
2,2022-08-08,"33,950.8783","28,429.9083","7,272.2309","6,327.3109",189,247,155,195,536.9000,"2,203.8500",0.0000,0.0000,"1,015.4900",0.0000,0.0000,583.0000,"2,071.0000",0.0000,0.0000,443.0000,0.0000,0.0000,"7,659.0000","352,515.0000",0.0000,0.0000,"83,984.0000",0.0000,0.0000,"1,276.0000",0.0000,"2,913.0000",907.0000,777.0000,264,2022,8,32,3,0,0,0
3,2022-08-15,"34,204.2105","29,107.4099","9,642.3842","9,307.1898",223,290,191,250,600.6600,"2,456.8100",0.0000,0.0000,"1,010.7800",0.0000,0.0000,756.0000,"2,131.0000",0.0000,0.0000,355.0000,0.0000,0.0000,"9,619.0000","401,306.0000",0.0000,0.0000,"97,093.0000",0.0000,0.0000,"1,483.0000",0.0000,"2,836.0000",994.0000,941.0000,461,2022,8,33,3,0,0,0
4,2022-08-22,"40,686.4562","34,453.4948","10,718.3154","7,921.3493",217,358,177,303,397.1200,"2,459.9100",0.0000,0.0000,"1,037.6900",0.0000,0.0000,632.0000,"2,359.0000",0.0000,0.0000,476.0000,0.0000,0.0000,"12,972.0000","332,852.0000",0.0000,0.0000,"109,117.0000",0.0000,0.0000,604.0000,0.0000,932.0000,270.0000,236.0000,7677,2022,8,34,3,0,0,0
5,2022-08-29,"86,141.5483","72,214.4594","32,978.3722","27,411.7660",503,683,420,541,479.7200,"2,256.8800",0.0000,0.0000,"1,103.5400",0.0000,0.0000,918.0000,"2,505.0000",0.0000,0.0000,438.0000,0.0000,0.0000,"14,775.0000","315,038.0000",0.0000,0.0000,"113,663.0000",0.0000,0.0000,"2,046.0000",0.0000,"3,484.0000","3,480.0000",986.0000,344,2022,8,35,3,0,0,0
6,2022-09-05,"68,237.7923","59,589.7491","22,781.3835","14,014.7173",385,550,326,441,609.4900,"2,938.6400",0.0000,0.0000,"1,270.6700",0.0000,0.0000,"1,350.0000","3,036.0000",0.0000,0.0000,634.0000,0.0000,0.0000,"33,843.0000","338,141.0000",2.0000,0.0000,"136,599.0000",0.0000,0.0000,"2,503.0000",0.0000,"3,503.0000","2,448.0000","1,474.0000",589,2022,9,36,3,0,0,0
7,2022-09-12,"19,098.6833","17,480.6933","3,977.6788","3,370.9488",165,235,146,211,613.6100,"2,192.4300",32.0700,0.0000,"1,187.4975",0.0000,0.0000,"1,408.0000","2,757.0000",35.0000,0.0000,468.0000,0.0000,0.0000,"42,759.0000","259,056.0000","3,483.0000",0.0000,"132,543.0000",0.0000,0.0000,"2,014.0000",0.0000,"3,011.0000",567.0000,"1,220.0000",286,2022,9,37,3,0,0,0
8,2022-09-19,"22,216.7812","20,405.8007","3,760.5983","3,462.1879",163,208,145,188,494.6100,"2,623.7800",404.3100,0.0000,"1,125.7100",0.0000,0.0000,900.0000,"3,045.0000",450.0000,0.0000,562.0000,0.0000,0.0000,"11,423.0000","296,560.0000","50,376.0000",0.0000,"81,811.0000",0.0000,0.0000,"1,657.0000",0.0000,"2,976.0000",453.0000,"1,229.0000",463,2022,9,38,3,0,0,0
9,2022-09-26,"36,680.8666","31,834.8715","2,206.5415","2,343.5836",209,264,179,223,394.7100,"1,849.3400",92.9900,0.0000,"1,082.9700",0.0000,0.0000,716.0000,"2,288.0000",399.0000,0.0000,405.0000,0.0000,0.0000,"12,174.0000","257,611.0000","170,157.0000",0.0000,"49,130.0000",0.0000,0.0000,"3,385.0000",0.0000,"2,5

---
## 1.8. SPEND SCALE VALIDATION

Before saving the clean dataset, we verify that all spend columns are in correct USD scale — i.e., real dollar amounts typical for a mid-size US DTC brand (~$500–$50,000 per week per active channel).

**What:** A final automated check that confirms the weekly spend values are in the expected USD range after all normalization steps.

**Why:** The sub-unit scaling correction in step 1.6.4 applies a conditional ÷ 1e9 to spend rows. If a column's distribution shifted unexpectedly — for example due to a change in the source file format — the resulting spend figures could still be in the wrong scale without triggering any earlier error. Catching this before the file is saved prevents the issue from propagating to downstream notebooks.

**Expected result:** All spend columns show `✓` with weekly mean values between $500 and $50,000. Any column flagged as `OUT OF RANGE` requires investigation of the normalization logic in section 1.6.4 before proceeding.

In [17]:
# ── Spend scale validation — run before saving the clean file ─────────────
SPEND_COLS_CHECK = ['spend_ps', 'spend_pmax', 'spend_fb', 'spend_ig',
                    'spend_display', 'spend_video']

print("=" * 65)
print("SPEND SCALE VALIDATION")
print("Expected range for a mid-size US DTC brand: $500 – $50,000 / week")
print("=" * 65)

all_ok = True
for col in SPEND_COLS_CHECK:
    if col not in df_weekly.columns:
        continue
    active = df_weekly[col][df_weekly[col] > 0]
    if len(active) == 0:
        print(f"  {col:<20s}  No active weeks — skipped")
        continue
    col_min, col_max, col_mean = active.min(), active.max(), active.mean()
    status = "✓" if 100 < col_mean < 500_000 else "⚠️  OUT OF RANGE"
    if "⚠️" in status:
        all_ok = False
    print(f"  {col:<20s}  mean=${col_mean:>10,.0f}  "
          f"min=${col_min:>8,.0f}  max=${col_max:>10,.0f}  {status}")

print()
if all_ok:
    print("✓  All spend columns are in correct USD scale — safe to save.")
else:
    print("⚠️  One or more columns may still be in sub-unit scale.")
    print("   Review the normalization logic in section 1.6.4 before proceeding.")
    raise ValueError("Spend scale validation failed — do not save until resolved.")

SPEND SCALE VALIDATION
Expected range for a mid-size US DTC brand: $500 – $50,000 / week
  spend_ps              mean=$       512  min=$      26  max=$     2,933  ✓
  spend_pmax            mean=$     2,539  min=$     352  max=$    11,706  ✓
  spend_fb              mean=$     1,974  min=$      11  max=$     6,065  ✓
  spend_ig              mean=$       575  min=$      11  max=$     1,932  ✓
  spend_display         mean=$       146  min=$      20  max=$       404  ✓
  spend_video           mean=$       111  min=$      14  max=$       203  ✓

✓  All spend columns are in correct USD scale — safe to save.


---
## 1.9. SAVE CLEAN DATA

We save the clean weekly dataset with a date stamp in the filename for traceability.

**Storage decision:** `.xlsx` format is used for compatibility with business stakeholder review. A `.parquet` version would be preferred for large-scale pipelines but is not required at this dataset size (74 rows).

In [18]:
# Save clean dataset with date stamp
process_date = datetime.now().strftime('%Y%m%d')
FILE_INTERIM = PATH_DATA_INTERIM / f'hairbright_clean_{process_date}.xlsx'

df_weekly.to_excel(FILE_INTERIM, index=False)

print(f"Clean dataset saved:")
print(f"  Path  : {FILE_INTERIM}")
print(f"  Shape : {df_weekly.shape[0]} rows × {df_weekly.shape[1]} columns")
print(f"  Size  : {FILE_INTERIM.stat().st_size / 1024:.1f} KB")

Clean dataset saved:
  Path  : /content/drive/MyDrive/PRO/+ DATA SCIENCE/TFM/marketing-mix-modeling-beauty/data/interim/hairbright_clean_20260415.xlsx
  Shape : 74 rows × 43 columns
  Size  : 22.8 KB


---
## 1.10. TRANSFORMATION SUMMARY

### 1.10.1. Changes applied

| Step | Transformation | Detail |
|:-----|:---------------|:-------|
| 1.6.1 | Drop metadata columns | 8 identifier columns removed |
| 1.6.1 | Drop zero-coverage channels | Google Shopping + TikTok (100% null) |
| 1.6.2 | Fix corrupted spend values | 7 date-object errors in `GOOGLE_PAID_SEARCH_SPEND` → imputed with median |
| 1.6.3 | Coerce all numeric columns | All spend, clicks, impressions and monetary columns → `float64`, NaN → 0 |
| 1.6.4 | Normalize sub-unit scaled fields | ÷ 1e9 applied to 4 revenue/discount columns; conditional ÷ 1e9 to 6 spend columns |
| 1.6.5 | Weekly aggregation | 509 daily rows → 74 weekly rows (sum) |
| 1.6.6 | Column renaming | snake_case standardization across all columns |
| 1.6.7 | Time flags | `is_q4`, `is_bf_week`, `is_holiday` added |
| 1.8 | Spend scale validation | All spend columns confirmed in correct USD range before save |

In [19]:
# Final summary
print("=" * 60)
print("CLEANING PROCESS SUMMARY")
print("=" * 60)
print(f"\nOriginal dataset : {initial_records} daily rows")
print(f"Clean dataset    : {len(df_weekly)} weekly rows")
print(f"Columns          : {df_raw.shape[1]} raw → {df_weekly.shape[1]} clean")
print(f"Revenue range    : ${df_weekly['revenue'].min():,.0f} – ${df_weekly['revenue'].max():,.0f} USD/week")
print(f"Total revenue    : ${df_weekly['revenue'].sum():,.0f} USD ({len(df_weekly)} weeks)")
print(f"Total ad spend   : ${df_weekly[[c for c in df_weekly.columns if c.startswith('spend_')]].sum().sum():,.0f} USD")
print(f"Period covered   : {df_weekly['week'].min().date()} → {df_weekly['week'].max().date()}")
print(f"\nData quality     : 0 nulls | 0 duplicate weeks ✓")
print(f"Output file      : data/interim/hairbright_clean_YYYYMMDD.xlsx")

CLEANING PROCESS SUMMARY

Original dataset : 509 daily rows
Clean dataset    : 74 weekly rows
Columns          : 50 raw → 43 clean
Revenue range    : $10,340 – $474,930 USD/week
Total revenue    : $4,183,141 USD (74 weeks)
Total ad spend   : $327,225 USD
Period covered   : 2022-07-25 → 2023-12-18

Data quality     : 0 nulls | 0 duplicate weeks ✓
Output file      : data/interim/hairbright_clean_YYYYMMDD.xlsx


### 1.10.2. Reusability note

**Reusable pipeline for monthly updates.** When new monthly data is appended to the raw file:

1. Re-run this notebook from top to bottom.
2. The corrupted spend detection (1.6.2) will catch any new date-object errors automatically.
3. The weekly aggregation window will extend to include new weeks.
4. The output filename is date-stamped — previous clean files are preserved.

---

**Next notebook:** `02_eda_en.ipynb` — Exploratory Data Analysis, distribution analysis, correlation matrix, adstock visualization and outlier detection.